In [1]:
from scripts.Utils import TempRel_Utils
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizerFast, TrainingArguments, Trainer, DataCollatorWithPadding
from scripts.Reader import obtain_dataset, id_token_labels
import datasets
import os

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda')
print("Current Device:", torch.cuda.current_device(), torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Current Device: 0 NVIDIA GeForce RTX 4070 Ti


In [ ]:
datasets, label_list, label2id, id2label = obtain_dataset("TBDense", "E-T")

In [ ]:
model_name = 'roberta-large'
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=len(label_list), label2id=label2id, id2label=id2label)
tokenizer.add_tokens(["[ES]","[EE]","[TS]","[TE]"])
model.resize_token_embeddings(len(tokenizer))
utils = TempRel_Utils(tokenizer, label_list)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [4]:
datasets = utils.tokenize_datasets(datasets)

Map:   0%|          | 0/4429 [00:00<?, ? examples/s]

Map:   0%|          | 0/1515 [00:00<?, ? examples/s]

Map:   0%|          | 0/669 [00:00<?, ? examples/s]

In [5]:
training_args = TrainingArguments(
    output_dir="./results/E-E-TempRel",
    logging_dir="./logs/E-E-TempRel",
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=100,
    num_train_epochs=10,
    save_total_limit=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [6]:
event_event_temprel = Trainer(
    model=model,
    args=training_args,
    compute_metrics=utils.compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    tokenizer=tokenizer,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
)

C:\Users\Harry\AppData\Local\Temp\ipykernel_5564\1315440943.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  event_event_temprel = Trainer(


In [7]:
event_event_temprel.train()

Step,Training Loss,Validation Loss,Precision,Recall,F1
100,1.509600,1.494745,0.379671,0.379671,0.379671
200,1.477200,1.483526,0.379671,0.379671,0.379671
300,1.472900,1.439295,0.379671,0.379671,0.379671
400,1.433600,1.520435,0.379671,0.379671,0.379671
500,1.465600,1.510959,0.379671,0.379671,0.379671
600,1.453400,1.504221,0.379671,0.379671,0.379671
700,1.461300,1.479638,0.379671,0.379671,0.379671
800,1.445900,1.495642,0.379671,0.379671,0.379671
900,1.460100,1.464475,0.379671,0.379671,0.379671
1000,1.451700,1.504992,0.379671,0.379671,0.379671


KeyboardInterrupt: 

In [ ]:
event_event_temprel.evaluate(datasets["test"])

In [ ]:
event_event_temprel.save_model("./results/E-E-TempRel/final_model")